In [ ]:
import scanpy as sc
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import scipy


In [ ]:
data_annotated_path = "/home/bmb/haxx/working/ceisel_mumm/"
# data = sc.read_h5ad(data_annotated_path + "joint_annotated.h5ad")
data = sc.read_h5ad(data_annotated_path + "full_annotations_leiden.h5ad")

data.shape

# Building A Timeline

In [ ]:
plt.figure()
plt.title("Log total counts vs time")
plt.boxplot(
    [data.obs['log_total_counts'][data.obs['time'] == t] for t in [12,24,48,72]],
    showfliers=False,
)
plt.xticks(
    [1,2,3,4],
    ["12h","24h","48h","72h"]
)
plt.ylabel("Log total counts")
plt.show()


# PCA IQR Timeline, Big Sheet

We can briefly look at the timecourse of each individual PC across the entire dataset with no filtering to sanity check our data and see the overall level of stability. We also want to observe whether or not there are obvious outliers

In [ ]:
def extract_iqrs(data,low_percentile=25,high_percentile=75):
    return {
        'low':np.percentile(data,low_percentile),
        'median':np.median(data),
        'high':np.percentile(data,high_percentile),
    }

def extract_sems(data,sem_multiple=2):
    sem = scipy.stats.sem(data)
    mean = np.mean(data)
    return {
        'low':mean - (sem*sem_multiple),
        'mean':mean,
        'high':mean + (sem*sem_multiple),
    }

def filled_centerline_to_ax(ax,times,iqrs,label=None,fill_label=None,ax_label=None,central_tendency=None,**kwargs):
    if central_tendency is None:
            all_keys = {key for iqr in iqrs.values() for key in iqr.keys()}
            candidates = all_keys & {'median', 'mean'}
            
            match len(candidates):
                case 1: central_tendency = candidates.pop()
                case 2: raise KeyError(f"Ambiguous central tendencies: {all_keys}")
                case _: raise KeyError(f"No central tendency found: {all_keys}")

    ax.plot(
        times,
        [iqrs[t][central_tendency] for t in times],
        label=label,
        **kwargs
    )
    ax.fill_between(
        times,
        [iqrs[t]['low'] for t in times],
        [iqrs[t]['high'] for t in times],
        label=fill_label,alpha=.2
    )

    ax.set_xlabel("Time (h)")
    ax.set_ylabel(ax_label)
    ax.set_title(ax_label)
    if label or fill_label:
        ax.legend()

In [ ]:
control_mask =  data.obs['exp_condition'] == "Cntr"
mutant_mask = data.obs['exp_condition'] == "Mtz"

times = [12,24,48,72]

time_masks = [
    data.obs['time'] == t
    for t in times
]

fig,axes = plt.subplots(5,10,figsize=(30,15))
for i in range(50):

    time_control_iqrs = {}
    time_mutant_iqrs = {}
    for t,time_mask in zip(times,time_masks):
        combined_control_mask = control_mask & time_mask
        combined_mutant_mask = mutant_mask & time_mask

        control_target = data.obsm['X_pca'][:,i][combined_control_mask]
        mutant_target = data.obsm['X_pca'][:,i][combined_mutant_mask]

        time_control_iqrs[t] = extract_iqrs(control_target)
        time_mutant_iqrs[t] = extract_iqrs(mutant_target)
    
    ax = axes[i//10,i%10]

    filled_centerline_to_ax(
        ax,times,time_control_iqrs,
        # label="control",fill_label="control (IQR)",
        ax_label=f"PC{i+1}",
    )
    filled_centerline_to_ax(
        ax,times,time_mutant_iqrs,
        #label="mutant",fill_label="mutant (IQR)",
        ax_label=f"PC{i+1}",
    )

fig.tight_layout()
fig.show()

# Cell Death Scoring

The particulars of producing these scores are in the sister file "annotations", but this is the set of gene sets used by the TBI guys. They can be updated with alternatives on demand. (Swap in the gene sets and map the homologs to zebrafish genome)

We want to plot the timecourse of the score, as well as the scores over the UMAP to see if there are any obvious patterns.

Brief note: representing an uncertainty score here is a little fraught, I'm not a huge fan of either the IQR or the SEM, the former is too vague and the latter is too conservative (as written it is treating each cell as a unique observation)

I want to check if there are biological replicate annotations, if so we can re-calculate the SEM on a per-sample basis, which will better represent the actual variation certainty. Please stand by for this.


In [ ]:
sc.tl.score_genes(data,data.var_names[data.var['parthanatos']],score_name="parthanatos")
sc.tl.score_genes(data,data.var_names[data.var['necroptosis']],score_name="necroptosis")
sc.tl.score_genes(data,data.var_names[data.var['apoptosis']],score_name="apoptosis")


In [ ]:
# Let's plot a timecourse for each between mutant and ctrl 

times = [12,24,48,72]

time_masks = [
    data.obs['time'] == t
    for t in times
]

# Timecourse with an IQR shadow

fig,axes = plt.subplots(1,3,figsize=(15,5))
for i,score in enumerate(['parthanatos','necroptosis','apoptosis']):

    # We build the IQR shadows here as dictionaries

    time_control_iqrs = {}
    time_mutant_iqrs = {}

    for t,time_mask in zip(times,time_masks):
        combined_control_mask = control_mask & time_mask
        combined_mutant_mask = mutant_mask & time_mask

        control_data = data.obs[score][combined_control_mask]
        mutant_data = data.obs[score][combined_mutant_mask]

        time_control_iqrs[t] = extract_iqrs(control_data)
        time_mutant_iqrs[t] = extract_iqrs(mutant_data)

    # Actual plotting
    ax = axes[i]
    filled_centerline_to_ax(
        ax,times,time_control_iqrs,
        label="control (median)",fill_label="control (IQR)",ax_label=f"{score}, (all cells)",
    )
    filled_centerline_to_ax(
        ax,times,time_mutant_iqrs,
        label="mutant (median)",fill_label="mutant (IQR)",ax_label=f"{score}, (all cells)",
    )
    ax.legend(loc="upper left")

fig.tight_layout()
fig.show()

# Timecourse with an SEM 

fig,axes = plt.subplots(1,3,figsize=(15,5))
for i,score in enumerate(['parthanatos','necroptosis','apoptosis']):

    # Build SEM shadows

    time_control_sem_range = {}
    time_mutant_sem_range = {}

    for t,time_mask in zip(times,time_masks):
        combined_control_mask = control_mask & time_mask
        combined_mutant_mask = mutant_mask & time_mask

        control_target = data.obs[score][combined_control_mask]
        mutant_target = data.obs[score][combined_mutant_mask]

        time_control_sem_range[t] = extract_sems(control_target)
        time_mutant_sem_range[t] = extract_sems(mutant_target)

    # Actual plotting
    ax = axes[i]
    filled_centerline_to_ax(
        ax,times,time_control_sem_range,
        label="control (mean)",fill_label="control (SEM)",ax_label=f"{score}, (all cells)",
    )
    filled_centerline_to_ax(
        ax,times,time_mutant_sem_range,
        label="mutant (mean)",fill_label="mutant (SEM)",ax_label=f"{score}, (all cells)",
    )
    ax.legend(loc="best")

fig.tight_layout()
fig.show()

### UMAP Score Plots

We can see that the distribution of the scores is similar but not fully overlapping. We can probably confirm this intuition with a bunch of pairwise scatters, but this basically tracks with the lit anyway afaik. 

In [ ]:
fig,axes = plt.subplots(1,3,figsize=(15,5))
axes = axes.flatten()

sc.pl.umap(data,color='parthanatos',ax=axes[0],show=False)
sc.pl.umap(data,color='necroptosis',ax=axes[1],show=False)
sc.pl.umap(data,color='apoptosis',ax=axes[2],show=False)

fig.show()

# Leiden 

We will use Leiden to do unsupervised partitioning and then look at the partitions to see how well they match up with already-known markers. This will let us easily pluck out eg RGCs without being too bothered about re-rolling a classifier. 

### Sundries

In [ ]:
# sc.tl.leiden(data, resolution=0.5)

In [ ]:
# sc.pl.umap(data, color=['leiden'],legend_loc='on data')

In [ ]:
# sc.tl.rank_genes_groups(data, 'leiden')
# sc.pl.rank_genes_groups(data, n_genes=25)


In [ ]:
# top_3_across_all = data.uns['rank_genes_groups']['names'][:3]
# top_3_across_all = np.array([[top_3_across_all[j][i] for i in range(20)] for j in range(3)])

In [ ]:
# top_3_across_all.flatten()

### Plot

In [ ]:
# sc.pl.dotplot(data, top_3_across_all.flatten(),groupby="leiden",figsize=(20,10))
sc.pl.dotplot(data, ['rbpms2a','rbpms2b'],groupby="leiden",figsize=(20,10))

Leiden is stochastic so we have run this once and saved it for posterity. To be clear: the partitions Leiden creates are highly stable, but they are discovered in random order, so there is always roughly the same population of cells identified as a cluster, but sometimes it is cluster 5 and sometimes it is cluster 15.

If you wish to re-run this analysis you need only use the marker gene dot plot to re-annotate the target leiden clusters that identify RGC subpops. This logic is in the next section (labeled RGC mask)

In [ ]:
sc.pl.umap(data,color='leiden',legend_loc='on data')

In [ ]:
# Commented out to avoid over-writing the frozen analysis
# sc.write("full_annotations_leiden.h5ad",data)

# Subset to Radial Glia

We want to check if the behavior of RGCs in particular is different than the rest of the pop with respect to the scores of interest.

We will isolate just the RGCs using leiden mapping, and then re-plot some of our previous analysis

In [ ]:
# Edit this if the leiden mappings shift because the analysis was re-run
rgc_mask = (data.obs['leiden'] == '3') | (data.obs['leiden'] == '10')
rgc = data[rgc_mask]
rgc

In [ ]:
# These timecourses are focused in on RGCs. 
# Second verse same as first

times = [12,24,48,72]

time_rgc_masks = [
    rgc.obs['time'] == t
    for t in times
]

control_rgc_mask =  rgc.obs['exp_condition'] == "Cntr"
mutant_rgc_mask = rgc.obs['exp_condition'] == "Mtz"

fig,axes = plt.subplots(1,3,figsize=(15,5))
for i,score in enumerate(['parthanatos','necroptosis','apoptosis']):

    time_control_rgc_iqrs = {}
    time_mutant_rgc_iqrs = {}

    for t,time_rgc_mask in zip(times,time_rgc_masks):

        combined_rgc_control_mask = control_rgc_mask & time_rgc_mask
        combined_rgc_mutant_mask = mutant_rgc_mask & time_rgc_mask

        time_control_iqrs[t] = extract_iqrs(rgc.obs[score][combined_rgc_control_mask])
        time_mutant_iqrs[t] = extract_iqrs(rgc.obs[score][combined_rgc_mutant_mask])

    ax = axes[i]
    
    filled_centerline_to_ax(
        ax,times,time_control_iqrs,
        label="control (median)",fill_label="control (IQR)",ax_label=f"{score}, (RGC)",
    )
    filled_centerline_to_ax(
        ax,times,time_mutant_iqrs,
        label="mutant (median)",fill_label="mutant (IQR)",ax_label=f"{score}, (RGC)",
    )

fig.tight_layout()
fig.show()

fig,axes = plt.subplots(1,3,figsize=(15,5))
for i,score in enumerate(['parthanatos','necroptosis','apoptosis']):

    time_control_rgc_sem_range = {}
    time_mutant_rgc_sem_range = {}

    for t,time_rgc_mask in zip(times,time_rgc_masks):

        combined_rgc_control_mask = control_rgc_mask & time_rgc_mask
        combined_rgc_mutant_mask = mutant_rgc_mask & time_rgc_mask

        time_control_sem_range[t] = extract_sems(rgc.obs[score][combined_rgc_control_mask])
        time_mutant_sem_range[t] = extract_sems(rgc.obs[score][combined_rgc_mutant_mask])

    ax = axes[i]
    
    filled_centerline_to_ax(
        ax,times,time_control_sem_range,
        label="control (mean)",fill_label="control (SEM)",ax_label=f"{score} (RGC)",
    )
    filled_centerline_to_ax(
        ax,times,time_mutant_sem_range,
        label="mutant (mean)",fill_label="mutant (SEM)",ax_label=f"{score} (RGC)",
    )

fig.tight_layout()
fig.show()

# RGC Vs Rest Comparison

Let's compare the RGCs to the rest of the population to observe any differences for our scores of interest

In [ ]:
non_rgc = data[~rgc_mask]

times = [12,24,48,72]

rgc_time_masks = [
    rgc.obs['time'] == t
    for t in times
]

non_rgc_time_masks = [
    non_rgc.obs['time'] == t
    for t in times
]

rgc_control_mask =  rgc.obs['exp_condition'] == "Cntr"
rgc_mutant_mask = rgc.obs['exp_condition'] == "Mtz"

non_rgc_control_mask =  non_rgc.obs['exp_condition'] == "Cntr"
non_rgc_mutant_mask = non_rgc.obs['exp_condition'] == "Mtz"

fig,axes = plt.subplots(1,3,figsize=(15,5))
for i,score in enumerate(['parthanatos','necroptosis','apoptosis']):

    time_rgc_control_iqrs = {}
    time_rgc_mutant_iqrs = {}
    time_non_rgc_control_iqrs = {}
    time_non_rgc_mutant_iqrs = {}

    for t,(rgc_time_mask,non_rgc_time_mask) in zip(times,zip(rgc_time_masks,non_rgc_time_masks)):

        combined_rgc_control_mask = rgc_control_mask & rgc_time_mask
        combined_rgc_mutant_mask = rgc_mutant_mask & rgc_time_mask

        combined_non_rgc_control_mask = non_rgc_control_mask & non_rgc_time_mask
        combined_non_rgc_mutant_mask = non_rgc_mutant_mask & non_rgc_time_mask

        time_rgc_control_iqrs[t] = extract_iqrs(rgc.obs[score][combined_rgc_control_mask])
        time_rgc_mutant_iqrs[t] = extract_iqrs(rgc.obs[score][combined_rgc_mutant_mask])

        time_non_rgc_control_iqrs[t] = extract_iqrs(non_rgc.obs[score][combined_non_rgc_control_mask])
        time_non_rgc_mutant_iqrs[t] = extract_iqrs(non_rgc.obs[score][combined_non_rgc_mutant_mask])

    ax = axes[i]

    # We're going to discard all the IQR stuff here because the shadows make this more or less unreadable. 
    # But I do think it's important to visualize this in overlay to convey the idea

    ax.plot(times, [time_rgc_control_iqrs[t]['median'] for t in times], label="RGC Control", color='r')
    ax.plot(times, [time_rgc_mutant_iqrs[t]['median'] for t in times], label="RGC Mutant", color='b')
    ax.plot(times, [time_non_rgc_control_iqrs[t]['median'] for t in times], label="Non-RGC Control", color='r', linestyle='--',alpha=.3)
    ax.plot(times, [time_non_rgc_mutant_iqrs[t]['median'] for t in times], label="Non-RGC Mutant", color='b', linestyle='--',alpha=.3)

    ax.set_xlabel("Time (h)")
    ax.set_ylabel(f"{score}")
    ax.set_title(f"{score} (RGC vs Non-RGC)")
    ax.legend()

fig.tight_layout()
fig.show()


# PCA sanity check

We should check if any computed PCs meaningfully correspond to the computed cell death scores, as this would be a simple way to verify whether or not this is a relatively orthogonal signal

In [ ]:
corr = np.corrcoef(data.obsm['X_pca'].T,data.obs[['parthanatos','necroptosis','apoptosis']].T)[50:,:50]

plt.figure()
plt.imshow(
    corr.T,
    cmap='bwr',vmin=-1,vmax=1,
    aspect='auto'
)
plt.ylabel("PCs")
plt.xticks(np.arange(3),labels=['parthanatos','necroptosis','apoptosis'])
plt.colorbar(label="Pearson Correlation")
plt.title("Cell Death Scores vs PCA Scores")
plt.show()

In [ ]:
parthanatos_min,parthanatos_max = np.min(corr[0]),np.max(corr[0])
necroptosis_min,necroptosis_max = np.min(corr[1]),np.max(corr[1])
apoptosis_min,apoptosis_max = np.min(corr[2]),np.max(corr[2])

parthanatos_min,parthanatos_max = np.around(parthanatos_min,3),np.around(parthanatos_max,3)
necroptosis_min,necroptosis_max = np.around(necroptosis_min,3),np.around(necroptosis_max,3)
apoptosis_min,apoptosis_max = np.around(apoptosis_min,3),np.around(apoptosis_max,3)

print(f"Correlations: \t\tMin\t\tMax")
print("========================================================")
print(f"parthanatos \t\t{parthanatos_min},\t\t{parthanatos_max}")
print(f"necroptosis \t\t{necroptosis_min},\t\t{necroptosis_max}")
print(f"apoptosis \t\t{apoptosis_min},\t\t{apoptosis_max}")

Apoptosis seems to have popped out as the strongest but not by a massive amount. It has (negative) 33% correlation to PC 2